In [9]:
import pandas as pd
import sqlite3

<h2>Поместите базу данных в подпапку data в каталоге src.</h2>

<h2>Создайте подключение к базе данных с помощью библиотеки sqlite3.</h2>

In [10]:
conn = sqlite3.connect('../data/checking-logs.sqlite')

<h2>Получите схему таблицы, pageviews используя pd.io.sql.read_sql() и запрос "PRAGMA table_info(pageviews);".</h2>

In [11]:
schema = pd.io.sql.read_sql("PRAGMA table_info(pageviews);", conn)
print(schema)

   cid      name       type  notnull dflt_value  pk
0    0     index    INTEGER        0       None   0
1    1       uid       TEXT        0       None   0
2    2  datetime  TIMESTAMP        0       None   0


<h2>Возьмите только первые десять строк таблицы, pageviews чтобы увидеть, как она выглядит.</h2>

In [12]:
preview = pd.read_sql("SELECT * FROM pageviews LIMIT 10;", conn)
print(preview)

   index      uid                    datetime
0      0  admin_1  2020-04-17 12:01:08.463179
1      1  admin_1  2020-04-17 12:01:23.743946
2      2  admin_3  2020-04-17 12:17:39.287778
3      3  admin_3  2020-04-17 12:17:40.001768
4      4  admin_1  2020-04-17 12:27:30.646665
5      5  admin_1  2020-04-17 12:35:44.884757
6      6  admin_1  2020-04-17 12:35:52.735016
7      7  admin_3  2020-04-17 12:36:21.401412
8      8  admin_3  2020-04-17 12:36:22.023355
9      9  admin_1  2020-04-17 13:55:19.129243


<h2>Получите подтаблицу с помощью одного запроса, где:</h2>

- Используются только «uid» и «datetime».

- Используются только данные пользователя ( user_*), а не данные администратора.

- Сортируется по «uid» в порядке возрастания.

- Индексный столбец — «datetime».

- «datetime» преобразуется в DatetimeIndex.

- Имя фрейма данных — pageviews.

In [13]:
query = """
         SELECT uid, datetime
         FROM pageviews 
         WHERE uid LIKE 'user_%'
         ORDER BY CAST(SUBSTR(uid, 6) AS INT) ASC;
         """


In [14]:
pageviews = pd.read_sql(query, conn, index_col='datetime', parse_dates=['datetime'])

print(pageviews)

                                uid
datetime                           
2020-04-26 21:53:59.624136   user_1
2020-04-26 22:06:19.478143   user_1
2020-04-26 22:12:09.614497   user_1
2020-04-30 19:29:01.831635   user_1
2020-05-05 20:26:32.894852   user_1
...                             ...
2020-05-21 01:46:08.730098  user_28
2020-05-21 18:45:20.441142  user_28
2020-04-17 22:46:26.785035  user_30
2020-04-29 16:51:21.877630  user_30
2020-05-09 20:30:47.034282  user_30

[987 rows x 1 columns]


In [15]:
pageviews.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 987 entries, 2020-04-26 21:53:59.624136 to 2020-05-09 20:30:47.034282
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   uid     987 non-null    object
dtypes: object(1)
memory usage: 15.4+ KB


In [16]:
conn.close()